## Loading data

In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/bank_notes/plsda/bank_notes.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1':'26.07']

In [2]:
# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'26.07'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'26.07'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 08:18:06,653 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 08:18:07,246 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [3]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

In [4]:
# Covariância global entre cada variável espectral e a predição contínua
cov_scores = []
y_pred_vals = y_pred_cont.values
for col in Xcalclass_prep.columns:
    x_vals = Xcalclass_prep[col].values
    cov = np.cov(x_vals, y_pred_vals)[0, 1]
    cov_scores.append(cov)
cov_scores_df = pd.DataFrame(np.abs(cov_scores), index=Xcalclass_prep.columns, columns=['Covariance'])
cov_scores_df.plot()

## Spectral cuts (domain knowledge)

In [5]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 2.74),
('Ar ka + Ag L', 2.76, 3.47),
('Ca ka', 3.5, 3.91),
('Ca kb', 3.93, 4.24),
('Ti ka', 4.26, 4.72),
('Ti kb', 4.75, 5.13),
('background2', 5.16, 6.12),
('Fe ka', 6.15, 6.76),
('Fe kb', 6.79, 7.32),
('background3', 7.35, 7.78),
('Cu', 7.81, 8.29),
('background4', 8.32, 21.46),
('Ag ka scattering', 21.49, 22.71),
('background5', 22.74, 24.52),
('background6', 24.55, 26.07),
]

import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

## VIP, Regression Coefficients e SHAP (como no original)

In [6]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

# # vamos agora extrair as variaveis mais importantes atraves do método SHAP
# import shap

# # Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
# explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
# shap_values_pls = explainer_pls(Xcalclass_prep)

# shap_global_importance = pd.DataFrame({
#     'energy': Xpredclass_prep.columns,
#     'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
# shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# # vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
# energy_to_zone_shap = {}
# for zone_name, start, end in spectral_cuts:
#     for i in shap_global_importance['energy']:
#         i_float = float(i)
#         if start <= i_float <= end:
#             energy_to_zone_shap[i] = zone_name
# shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# # agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
# shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
# shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# shap_unique_df.to_csv('shap_bank_notes.csv', index=False, sep=';')
shap_unique_df = pd.read_csv('shap_bank_notes.csv', sep=';') # loading previously saved shap_unique_df

# **Comparando com o bagging**

In [7]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 42]

all_results = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    mi_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance',
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results[seed] = {
        'bags_result': bags_result_seed,
        'mi_results_dict': mi_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results[seed]['bags_result'],
        mi_results_dict=all_results[seed]['mi_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_by_seed[seed] = lrc_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_df_seed = lrc_by_seed[seed].rename(columns={'Node': f'Predicate_Seed_{seed}'})
    lrc_all_seeds_df = pd.concat([lrc_all_seeds_df, lrc_df_seed[[f'Predicate_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_unique_by_seed = {}
for seed, lrc_df in lrc_by_seed.items():
    lrc_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_unique_df = lrc_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_unique_by_seed[seed] = lrc_unique_df

lrc_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartado

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_Seed_0,Predicate_Seed_1,Predicate_Seed_42
0,Fe ka > -9.53,Fe ka > -9.53,Fe ka > -9.53
1,Fe ka > -9.80,Fe ka > -9.80,Fe ka > -9.80
2,Ca ka > -11.34,Fe ka > -10.00,Fe ka > -10.00
3,Fe ka > -10.00,Ca ka > -11.34,Ca ka > -9.12
4,Ca ka > -3.71,Ti ka <= 9.71,Fe kb > -3.35
...,...,...,...
89,Fe ka <= -10.00,Cu <= -4.31,background1 <= 2.22
90,background6 > 2.71,background3 > 2.09,Fe ka <= -10.00
91,Class_A,background2 > 2.59,Class_A
92,Class_B,Class_A,Class_B


In [8]:
import ks_folding as ksf

folds_result = ksf.kfold_predicates_roundrobin(
    zone_sums_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont,
    predicates_df=predicates_quantiles[0],
    k_folds=2,
    min_samples_ratio=0.001,  # 60% das amostras do fold
    verbose=True,
    per_predicate=True # escolhe entre fazer o fold por predicado ou globalmente (que faz o fold para todos os predicados juntos)
)

# Adiciona classe prevista (A/B) em cada DataFrame de predicado
for fold_name, pred_dict in folds_result.items():
    for rule, df_info in pred_dict.items():
        df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

mi_results_dict = exp.calculate_predicate_metrics(
    bags_result=folds_result,
    metric='covariance',
    threshold=0.001,
    #n_neighbors=5
)    

max_len = max(len(mi_df['Predicate']) for mi_df in mi_results_dict.values())
padded_dict = {
    f'Predicate_{fold}': list(mi_df['Predicate']) + [None]*(max_len - len(mi_df['Predicate']))
    for fold, mi_df in mi_results_dict.items()
}
all_cov_results = pd.DataFrame(padded_dict)
all_cov_results

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:18,520 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:18,524 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 2
Amostras por fold (aprox.): 142
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:18,574 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:18,578 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [27, 27]
  'background1 > -2.42': 230 amostras, folds: [115, 115]
  'background1 <= 2.22': 113 amostras, folds: [57, 56]
  'background1 > 2.22': 171 amostras, folds: [86, 85]
  'background1 <= 2.53': 171 amostras, folds: [86, 85]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001


,Predicate_Fold_1,Predicate_Fold_2
0,Fe ka > -9.25,Fe ka > -9.25
1,Fe ka > -9.53,Fe ka > -9.53
2,Fe ka > -9.80,Fe ka > -9.80
3,Fe ka > -10.00,Ca ka > -11.34
4,Ca ka > -9.12,Fe kb > -3.14
...,...,...
114,Fe kb <= -3.35,background5 <= -2.53
115,background5 <= 2.03,Ti kb <= -4.43
116,background1 > 2.22,Ca kb <= -4.15
117,None,Ca ka <= -11.34


In [9]:
DG = exp.build_fold_predicate_graph(
    bags_result=folds_result,           # Resultado dos folds (KS + Round-Robin)
    mi_results_dict=mi_results_dict,    # Rankings de Covariância por fold
    predicates_df=predicates_quantiles[0],  # DataFrame com metadados dos predicados
    random_state=42,                    # Semente para reprodutibilidade
    show_details=True,                  # Mostra detalhes da resolução
    normalize_weights=True,            # False = peso inteiro | True = peso [1/k, 1]
    weight_mode='cooccurrence',         # 'ranking' ou 'cooccurrence'
    co_occurrence_matrix=co_occurrence_matrix_df,  # Necessário se weight_mode='cooccurrence'
    apply_confidence_multiplier=True,    # True = peso × score | False = só co-ocorrência
    accumulate_cooccurrence_weights=True  # Nova opção!
)

# DG = exp.build_predicate_graph(
#     bags_result=folds_result,
#     mi_results_dict=mi_results_dict,
#     co_occurrence_matrix_df=co_occurrence_matrix_df,
#     predicates_df=predicates_quantiles[0],
#     random_state=42,
#     show_details=True
# )

# Calcula LRC para cada nó e compõe DataFrame
lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
# Seleciona apenas uma ocorrência por zona (maior LRC)
lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_ks_df

 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 231

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontrados: 4
Critério de desempate: PESO ACUMULADO (soma das co-ocorrências locais)

[Ca ka > -3.71 ↔ Fe kb > -3.35]  EMPATE (peso=53.00)
  ✗ Removida (aleatório): Ca ka > -3.71 → Fe kb > -3.35
  ✓ Mantida:  Fe kb > -3.35 → Ca ka > -3.71

[Ti ka <= 9.71 ↔ Ti ka > -11.81]  EMPATE (peso=168.00)
  ✗ Removida (aleatório): Ti ka > -11.81 → Ti ka <= 9.71
  ✓ Mantida:  Ti ka <= 9.71 → Ti ka > -11.81

[Ti kb <= 2.64 ↔ Ti ka > -3.08]  EMPATE (peso=60.00)
  ✗ Removida (aleatório): Ti ka > -3.08 → Ti kb <= 2.64
  ✓ Mantida:  Ti kb <= 2.64 → Ti ka > -3.08

[Ca kb <= -4.1

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Fe ka > -9.25,3.257238,Fe ka,-9.25,>
1,Fe ka > -9.53,3.191665,Fe ka,-9.53,>
2,Fe ka > -10.00,2.684141,Fe ka,-10.00,>
3,Fe ka > -9.80,2.639126,Fe ka,-9.80,>
4,Ca ka > -11.34,2.501105,Ca ka,-11.34,>
...,...,...,...,...,...
117,Ca ka <= -11.34,0.430745,Ca ka,-11.34,<=
118,Ca kb <= -4.15,0.391450,Ca kb,-4.15,<=
119,Ti ka <= -11.81,0.000041,Ti ka,-11.81,<=
120,Class_A,0.000000,None,None,None


# **Variando o numero de folds - modo cooccurrence**

In [10]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5, 6]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}
all_fold_results_cooc = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

        # Salvar no dicionário principal
    all_fold_results_cooc[k_folds] = {
        'cov_results_dict': mi_results_dict
    }

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='cooccurrence',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=True
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:21,340 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:21,343 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 2
Amostras por fold (aprox.): 142
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:21,564 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:21,571 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [27, 27]
  'background1 > -2.42': 230 amostras, folds: [115, 115]
  'background1 <= 2.22': 113 amostras, folds: [57, 56]
  'background1 > 2.22': 171 amostras, folds: [86, 85]
  'background1 <= 2.53': 171 amostras, folds: [86, 85]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 231

RESOLUÇÃO DE ARESTAS

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:24,186 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:24,188 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and

...

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:34,207 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:34,210 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and


Processando k_folds: 6

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 6
Amostras por fold (aprox.): 47
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:34,414 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:34,426 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 6

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [9, 9, 9, 9, 9, 9]
  'background1 > -2.42': 230 amostras, folds: [39, 39, 38, 38, 38, 38]
  'background1 <= 2.22': 113 amostras, folds: [19, 19, 19, 19, 19, 18]
  'background1 > 2.22': 171 amostras, folds: [29, 29, 29, 28, 28, 28]
  'background1 <= 2.53': 171 amostras, folds: [29, 29, 29, 28, 28, 28]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
  Fold_3: 120 predicados
  Fold_4: 120 predicados
  Fold_5: 120 predicados
  Fold_6: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de 

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



In [11]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
     features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values
features_importance

# vamos exportar o df features_importance para um arquivo excel onde vamos nomear a sheet de acordo com as comparacoes feitas
features_importance.to_excel('features_importance_bank_notes.xlsx', index=False, sheet_name='KS_fold_cooc')

In [12]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results.to_excel('rbo_bank_notes.xlsx', index=False, sheet_name='KS_fold_cooc')
rbo_results


,Reference,Method,RBO_Score
8,Vip,LRC_Seed_1,0.922269
4,Vip,LRC_kfold_4,0.915076
7,Vip,LRC_Seed_0,0.913865
9,Vip,LRC_Seed_42,0.873269
2,Vip,LRC_kfold_2,0.862955
5,Vip,LRC_kfold_5,0.768269
1,Vip,Shap,0.464014
3,Vip,LRC_kfold_3,0.399591
6,Vip,LRC_kfold_6,0.265734
0,Vip,Reg_coef,0.235755


# **Variando o numero de folds - modo ranking**

In [13]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5, 6]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}
all_fold_results_rank = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

        # Salvar no dicionário principal
    all_fold_results_rank[k_folds] = {
        'cov_results_dict': mi_results_dict
    }

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='ranking',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=True
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:38,770 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:38,774 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:38,884 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:38,894 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 2
Amostras por fold (aprox.): 142
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:38,996 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:39,001 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [27, 27]
  'background1 > -2.42': 230 amostras, folds: [115, 115]
  'background1 <= 2.22': 113 amostras, folds: [57, 56]
  'background1 > 2.22': 171 amostras, folds: [86, 85]
  'background1 <= 2.53': 171 amostras, folds: [86, 85]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se aplica ao modo 'cooccurrence')

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: RANKING

Folds processados: 2
Arestas criadas (a

...


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 5

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [11, 11, 11, 11, 10]
  'background1 > -2.42': 230 amostras, folds: [46, 46, 46, 46, 46]
  'background1 <= 2.22': 113 amostras, folds: [23, 23, 23, 22, 22]
  'background1 > 2.22': 171 amostras, folds: [35, 34, 34, 34, 34]
  'background1 <= 2.53': 171 amostras, folds: [35, 34, 34, 34, 34]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
  Fold_3: 120 predicados
  Fold_4: 120 predicados
  Fold_5: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências s

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:55,755 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:55,758 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Processando k_folds: 6

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 284
Número de folds: 6
Amostras por fold (aprox.): 47
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:55,959 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-19 08:18:55,980 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 120
Predicados válidos: 120
Predicados eliminados: 0
Folds criados: 6

Estatísticas por predicado válido:
  'background1 <= -2.42': 54 amostras, folds: [9, 9, 9, 9, 9, 9]
  'background1 > -2.42': 230 amostras, folds: [39, 39, 38, 38, 38, 38]
  'background1 <= 2.22': 113 amostras, folds: [19, 19, 19, 19, 19, 18]
  'background1 > 2.22': 171 amostras, folds: [29, 29, 29, 28, 28, 28]
  'background1 <= 2.53': 171 amostras, folds: [29, 29, 29, 28, 28, 28]
  ... e mais 115 predicados

Predicados por fold:
  Fold_1: 120 predicados
  Fold_2: 120 predicados
  Fold_3: 120 predicados
  Fold_4: 120 predicados
  Fold_5: 120 predicados
  Fold_6: 120 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='rank

In [14]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
    features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values

with pd.ExcelWriter('features_importance_bank_notes.xlsx', mode='a', engine='openpyxl') as writer:
    features_importance.to_excel(writer, sheet_name='KS_fold_rank', index=False)
features_importance

,Vip,Reg_coef,Shap,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5,LRC_kfold_6,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,Fe ka,Ti ka,Ti ka,Fe ka,Ca ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Ca ka,Ti kb,Ca ka,Ca ka,Fe ka,Ca ka,Ti ka,Ca ka,Ca ka,Ca ka,Ca ka
2,Ti ka,Ag ka scattering,Cu,Fe kb,Fe kb,Ti ka,Ca ka,Fe kb,Ti ka,Ti ka,Fe kb
3,Cu,Ca ka,Fe ka,Ti ka,Ti kb,Ca kb,Fe kb,Ca kb,Fe kb,Fe kb,Ti ka
4,Fe kb,Cu,Ti kb,Ca kb,Cu,Fe kb,Ti kb,Ti kb,Ca kb,Ca kb,Ca kb
5,Ca kb,background6,Ag ka scattering,Ti kb,Ca kb,Ti kb,Ca kb,Ti ka,Ti kb,Cu,Cu
6,Ti kb,Ca kb,background6,Cu,Ti ka,Cu,Cu,Cu,Cu,Ti kb,Ti kb
7,Ag ka scattering,background4,background4,background4,background4,background4,background4,background4,background5,background4,background4
8,background6,background2,background5,Ag ka scattering,background2,background5,background5,background5,background4,background5,background2
9,background4,Ar ka + Ag L,background1,background5,Ag ka scattering,background1,background2,background3,background2,background2,background5


In [15]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)

# Supondo que features_importance já está criado
with pd.ExcelWriter('rbo_bank_notes.xlsx', mode='a', engine='openpyxl') as writer:
    rbo_results.to_excel(writer, sheet_name='KS_fold_rank', index=False)
rbo_results

,Reference,Method,RBO_Score
8,Vip,LRC_Seed_1,0.922269
4,Vip,LRC_kfold_4,0.913865
7,Vip,LRC_Seed_0,0.913865
9,Vip,LRC_Seed_42,0.873269
2,Vip,LRC_kfold_2,0.867997
6,Vip,LRC_kfold_6,0.824734
5,Vip,LRC_kfold_5,0.808865
3,Vip,LRC_kfold_3,0.540351
1,Vip,Shap,0.464014
0,Vip,Reg_coef,0.235755


# **Comparando com rankings medios**

In [16]:
ranking_predicate_mean_seeds = {}
ranking_predicate_mean_unique_seeds = {}

for seed in random_seeds:
    mi_results_dict_seed = all_results[seed]['mi_results_dict']
    ranking_predicate_mean, ranking_predicate_mean_unique = exp.calculate_predicate_ranking_mean(
        mi_results_dict_seed, 
        return_unique_zones=True
    )
    ranking_predicate_mean_seeds[seed] = ranking_predicate_mean
    ranking_predicate_mean_unique_seeds[seed] = ranking_predicate_mean_unique
# Exibindo resultados para uma semente específica

In [17]:
ranking_predicate_mean_folds = {}
ranking_predicate_mean_unique_folds = {}

for fold in folds_list:
    mi_results_dict_fold = all_fold_results_cooc[fold]['cov_results_dict']
    ranking_predicate_mean, ranking_predicate_mean_unique = exp.calculate_predicate_ranking_mean(
        mi_results_dict_fold, 
        return_unique_zones=True
    )
    ranking_predicate_mean_folds[fold] = ranking_predicate_mean
    ranking_predicate_mean_unique_folds[fold] = ranking_predicate_mean_unique
# Exibindo resultados para uma semente específica

In [18]:
# Build the dictionary step by step to avoid mixing comprehension and static entries
features_dict = {
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
}
# Add the predicate rankings from seeds
for seed, ranking_df in ranking_predicate_mean_unique_seeds.items():
    features_dict[f'Cov_Mean_Seed_{seed}'] = ranking_df['Zone'].iloc[:10].values
# Add the predicate rankings from folds
for fold, ranking_df in ranking_predicate_mean_unique_folds.items():
    features_dict[f'Cov_Mean_Fold_{fold}'] = ranking_df['Zone'].iloc[:10].values
features_importance = pd.DataFrame(features_dict)
features_importance

,Vip,Reg_coef,Shap,Cov_Mean_Seed_0,Cov_Mean_Seed_1,Cov_Mean_Seed_42,Cov_Mean_Fold_2,Cov_Mean_Fold_3,Cov_Mean_Fold_4,Cov_Mean_Fold_5,Cov_Mean_Fold_6
0,Fe ka,Ti ka,Ti ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Ca ka,Ti kb,Ca ka,Ca ka,Ca ka,Ca ka,Ca ka,Ca ka,Fe kb,Ca ka,Ca ka
2,Ti ka,Ag ka scattering,Cu,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb,Ca ka,Fe kb,Fe kb
3,Cu,Ca ka,Fe ka,Ti ka,Ti ka,Ti ka,Ti ka,Ti ka,Ti ka,Ti ka,Ti ka
4,Fe kb,Cu,Ti kb,Ca kb,Ca kb,Ca kb,Ca kb,Ca kb,Ca kb,Ca kb,Ca kb
5,Ca kb,background6,Ag ka scattering,Cu,Cu,Cu,Cu,Cu,Cu,Cu,Cu
6,Ti kb,Ca kb,background6,Ti kb,Ti kb,Ti kb,Ti kb,Ti kb,Ti kb,Ti kb,Ti kb
7,Ag ka scattering,background4,background4,background4,background4,background4,background4,background4,background4,background4,background4
8,background6,background2,background5,background2,background2,background2,background2,background2,background2,background2,background1
9,background4,Ar ka + Ag L,background1,background5,background5,background5,background5,background5,background5,background1,background2


In [19]:
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'Cov_Mean_Seed_{seed}' for seed in random_seeds] + [f'Cov_Mean_Fold_{fold}' for fold in folds_list]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
2,Vip,Cov_Mean_Seed_0,0.873269
3,Vip,Cov_Mean_Seed_1,0.873269
4,Vip,Cov_Mean_Seed_42,0.873269
5,Vip,Cov_Mean_Fold_2,0.873269
6,Vip,Cov_Mean_Fold_3,0.873269
8,Vip,Cov_Mean_Fold_5,0.873269
9,Vip,Cov_Mean_Fold_6,0.873269
7,Vip,Cov_Mean_Fold_4,0.768269
1,Vip,Shap,0.464014
0,Vip,Reg_coef,0.235755


## modo ranking

In [20]:
ranking_predicate_mean_folds = {}
ranking_predicate_mean_unique_folds = {}

for fold in folds_list:
    mi_results_dict_fold = all_fold_results_rank[fold]['cov_results_dict']
    ranking_predicate_mean, ranking_predicate_mean_unique = exp.calculate_predicate_ranking_mean(
        mi_results_dict_fold, 
        return_unique_zones=True
    )
    ranking_predicate_mean_folds[fold] = ranking_predicate_mean
    ranking_predicate_mean_unique_folds[fold] = ranking_predicate_mean_unique
# Exibindo resultados para uma semente específica

In [21]:
# Build the dictionary step by step to avoid mixing comprehension and static entries
features_dict = {
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
}
# Add the predicate rankings from seeds
for seed, ranking_df in ranking_predicate_mean_unique_seeds.items():
    features_dict[f'Cov_Mean_Seed_{seed}'] = ranking_df['Zone'].iloc[:10].values
# Add the predicate rankings from folds
for fold, ranking_df in ranking_predicate_mean_unique_folds.items():
    features_dict[f'Cov_Mean_Fold_{fold}'] = ranking_df['Zone'].iloc[:10].values
features_importance = pd.DataFrame(features_dict)
features_importance

,Vip,Reg_coef,Shap,Cov_Mean_Seed_0,Cov_Mean_Seed_1,Cov_Mean_Seed_42,Cov_Mean_Fold_2,Cov_Mean_Fold_3,Cov_Mean_Fold_4,Cov_Mean_Fold_5,Cov_Mean_Fold_6
0,Fe ka,Ti ka,Ti ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Ca ka,Ti kb,Ca ka,Ca ka,Ca ka,Ca ka,Ca ka,Ca ka,Fe kb,Ca ka,Ca ka
2,Ti ka,Ag ka scattering,Cu,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb,Ca ka,Fe kb,Fe kb
3,Cu,Ca ka,Fe ka,Ti ka,Ti ka,Ti ka,Ti ka,Ti ka,Ti ka,Ti ka,Ti ka
4,Fe kb,Cu,Ti kb,Ca kb,Ca kb,Ca kb,Ca kb,Ca kb,Ca kb,Ca kb,Ca kb
5,Ca kb,background6,Ag ka scattering,Cu,Cu,Cu,Cu,Cu,Cu,Cu,Cu
6,Ti kb,Ca kb,background6,Ti kb,Ti kb,Ti kb,Ti kb,Ti kb,Ti kb,Ti kb,Ti kb
7,Ag ka scattering,background4,background4,background4,background4,background4,background4,background4,background4,background4,background4
8,background6,background2,background5,background2,background2,background2,background2,background2,background2,background2,background1
9,background4,Ar ka + Ag L,background1,background5,background5,background5,background5,background5,background5,background1,background2


In [22]:
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'Cov_Mean_Seed_{seed}' for seed in random_seeds] + [f'Cov_Mean_Fold_{fold}' for fold in folds_list]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
2,Vip,Cov_Mean_Seed_0,0.873269
3,Vip,Cov_Mean_Seed_1,0.873269
4,Vip,Cov_Mean_Seed_42,0.873269
5,Vip,Cov_Mean_Fold_2,0.873269
6,Vip,Cov_Mean_Fold_3,0.873269
8,Vip,Cov_Mean_Fold_5,0.873269
9,Vip,Cov_Mean_Fold_6,0.873269
7,Vip,Cov_Mean_Fold_4,0.768269
1,Vip,Shap,0.464014
0,Vip,Reg_coef,0.235755
